# 04 - Evaluation and Risk Decisions
Loads the saved model, scores the saved test set, and turns probabilities into Low / Medium / High risk decisions using the thresholds in `src/config.py`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))   # so we can import from src/

import joblib
import numpy as np
import pandas as pd
from src import config
from src.data_preprocessing import load_processed_data

model = joblib.load(config.MODEL_PATH)
_, X_test, _, y_test = load_processed_data()
y_pred_prob = model.predict_proba(X_test)[:, 1]
print("Thresholds -> Low <", config.LOW_RISK_MAX, "| High >=", config.HIGH_RISK_MIN)

In [ ]:
results = X_test.copy()
results["actual_default"] = y_test.values
results["default_probability"] = y_pred_prob
results["decision"] = results["default_probability"].apply(config.risk_band)
results[["actual_default", "default_probability", "decision"]].head()

In [ ]:
results["decision"].value_counts(normalize=True) * 100

In [ ]:
approval_rate = (results["decision"] == "Low Risk - Approve").mean()
rejection_rate = (results["decision"] == "High Risk - Reject").mean()
print(f"Approval Rate : {approval_rate*100:.2f}%")
print(f"Rejection Rate: {rejection_rate*100:.2f}%")

### Actual default rate inside each risk band
A good model has a low default rate in *Low Risk* and a high one in *High Risk*.

In [ ]:
results.groupby("decision")["actual_default"].agg(["count", "mean"]).rename(columns={"mean": "actual_default_rate"})

### Default recall at different thresholds

In [ ]:
from sklearn.metrics import recall_score, precision_score

for t in np.arange(0.2, 0.8, 0.05):
    preds = (y_pred_prob >= t).astype(int)
    print(f"Threshold {t:.2f} -> Recall: {recall_score(y_test, preds):.2f}  Precision: {precision_score(y_test, preds):.2f}")